# User's Guide: Generating the Dataset

This script is designed to generate the final cleaned dataset used in our project. Here we merge all the CSV files **exept from Monday** because it's only Benign traffic. We also delet the few windows were no attacks occurs for our next experiments. This code is a modified and compacted version of the work made by **Eric Anacleto Ribeiro** on Kaggle, I really advise you to check out his work if you want to see in more details all the preprocessing phase of the CIC-IDS-2017 dataset !

Links to his GitHub repositories :
- **Preprocessing details:** [GitHub - ml-intrusion-detection-cicids2017](https://github.com/anacletu/ml-intrusion-detection-cicids2017/blob/9b4f49f3234027e95660adb497a94eed9636f44c/cicids2017-comprehensive-data-processing-for-ml.ipynb)
- **Model Comparison:** [CICIDS2017 - ML Models Comparison: Supervised](https://github.com/anacletu/ml-intrusion-detection-cicids2017)

## 1. Setup on Kaggle

1.  Log in to [Kaggle](https://www.kaggle.com/).
2.  Create a **new Notebook**.
3.  In the right sidebar (or via the "Add Data" menu), search for and add the following dataset:
    * **Name:** `Network Intrusion dataset (CIC-IDS-2017)`
4.  Copy and paste the Python code below into the first cell and run it.

## 2. Retrieving the Data

1.  Once the script finishes (it may take a few minutes), go to the **Output** section (or `/kaggle/working` directory) of the notebook.
2.  Download the generated file: `cicids2017_cleaned_filtered.csv`.
3.  Place this file into the `data/` folder of the project.

In [1]:
import numpy as np
import pandas as pd
import os
import re  # Required for text cleaning (Regex)

# --- CONFIGURATION ---
# Time window size for analysis (number of packets/rows)
WINDOW_SIZE = 50000 
BENIGN_LABEL = 'BENIGN'
OUTPUT_FILENAME = 'cicids2017_cleaned_filtered.csv'

# --- 1. Loading and Merging Data (Chronological Order) ---

# Standard path on Kaggle
base_path = '/kaggle/input/' 

# Explicit list of files to load in exact chronological order
# (Monday is excluded as it contains only normal traffic)
files_in_order = [
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
]

dfs = []
print("--- STARTING DATA LOAD ---")

for filename in files_in_order:
    file_path = None
    # Recursive search for the file (Kaggle directory structure varies)
    for root, dirs, files in os.walk(base_path):
        if filename in files:
            file_path = os.path.join(root, filename)
            break
    
    if file_path:
        print(f"Loading: {filename}")
        dfs.append(pd.read_csv(file_path))
    else:
        print(f"WARNING: File not found -> {filename}")

# Concatenate all files into a single DataFrame
if dfs:
    data = pd.concat(dfs, axis=0, ignore_index=True)
    print(f"Total shape after merging: {data.shape}")
else:
    raise ValueError("Error: No files were loaded. Please check if the dataset is added correctly.")

# Free up memory
del dfs


# --- 2. Structural Cleaning ---

print("\n--- DATA CLEANING ---")

# Clean column names (remove extra spaces)
data.rename(columns={col: col.strip() for col in data.columns}, inplace=True)

# Remove strict duplicates
data = data.drop_duplicates(keep='first')

# Identify and remove columns containing identical data (duplicate columns)
print("Identifying redundant columns...")
identical_columns = {}
columns = data.columns.tolist()
seen = set()

for i in range(len(columns)):
    col1 = columns[i]
    if col1 in seen: continue
    
    for j in range(i + 1, len(columns)):
        col2 = columns[j]
        if col2 in seen: continue
        
        if data[col1].equals(data[col2]):
            identical_columns.setdefault(col1, []).append(col2)
            seen.add(col2)

for key, values in identical_columns.items():
    print(f" -> Removing copies of '{key}': {values}")
    data.drop(columns=values, inplace=True)

# Handle invalid values (Infinity and NaN)
print("Handling infinite and missing values...")
data.replace([np.inf, -np.inf], np.nan, inplace=True)
data = data.dropna()


# --- 3. Feature Selection ---

# Drop columns that have only one unique value (constants are useless for AI)
print("Dropping constant columns...")
unique_counts = data.nunique()
cols_to_drop = unique_counts[unique_counts == 1].index.tolist()
data.drop(columns=cols_to_drop, inplace=True)


# --- 4. Attack Label Correction ---

print("\n--- PROCESSING ATTACK TYPES ---")
data.rename(columns={'Label': 'Attack Type'}, inplace=True)

def clean_attack_label(label):
    """
    Cleans poorly encoded labels, specifically 'Web Attacks' 
    which often contain invisible non-ASCII characters.
    """
    if not isinstance(label, str):
        return str(label)
    
    # Replace any non-standard character with a hyphen
    label = re.sub(r'[^\x00-\x7F]+', '-', label)
    return label.strip()

data['Attack Type'] = data['Attack Type'].apply(clean_attack_label)

print("Detected Attack Types:")
print(data['Attack Type'].value_counts())


# --- 5. Intelligent Window Filtering ---
# This step reduces dataset size by removing sequences (windows)
# where absolutely nothing happens (100% normal traffic).

print(f"\n--- WINDOW FILTERING (Size: {WINDOW_SIZE}) ---")
initial_shape = data.shape

# 1. Reset index to ensure contiguous rows
data = data.reset_index(drop=True)

# 2. Create window identifiers (batch IDs)
data['temp_batch_id'] = data.index // WINDOW_SIZE

# 3. Identify windows containing AT LEAST ONE attack
#    (We do not want to train the model on purely benign windows that do not help detect anomalies)
attack_batches = data[data['Attack Type'] != BENIGN_LABEL]['temp_batch_id'].unique()

print(f"Total number of windows: {data['temp_batch_id'].max() + 1}")
print(f"Windows kept (containing at least 1 attack): {len(attack_batches)}")

# 4. Keep only the relevant windows
data = data[data['temp_batch_id'].isin(attack_batches)]

# 5. Final cleanup
data = data.drop(columns=['temp_batch_id'])
data = data.reset_index(drop=True)

final_shape = data.shape
rows_removed = initial_shape[0] - final_shape[0]
print(f"\nRows removed (100% Benign windows): {rows_removed}")
print(f"Final data shape: {final_shape}")


# --- 6. Export ---

print(f"\n--- EXPORTING ---")
print(f"Generating file {OUTPUT_FILENAME}...")
data.to_csv(OUTPUT_FILENAME, index=False)
print("Processing completed successfully.")

--- STARTING DATA LOAD ---
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Wednesday-workingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Total shape after merging: (2300825, 79)

--- DATA CLEANING ---
Identifying redundant columns...
 -> Removing copies of 'Total Fwd Packets': ['Subflow Fwd Packets']
 -> Removing copies of 'Total Backward Packets': ['Subflow Bwd Packets']
 -> Removing copies of 'Fwd Packet Length Mean': ['Avg Fwd Segment Size']
 -> Removing copies of 'Fwd PSH Flags': ['SYN Flag Count']
 -> Removing copies of 'Bwd PSH Flags': ['Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
 -> Removing cop